In [2]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
# from langchain_cohere import Cohere
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

c:\Users\palke\Desktop\Sample_Rag\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
## SETUP COMPANY DATA

chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")

Created 60 sample chunks for demonstration


In [4]:
documents=[Document(page_content=chunk , metadata={"source" : f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

In [5]:
#Vector Retriever setup
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(
    documents = documents,
      embedding=embedding_model,
        collection_metadata={"hnsw:space": "cosine" }

)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 20})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2967.92it/s]


In [6]:
#BM25 retriever setup
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k=20


In [7]:
#Hybrid Retriever setup
hybrid_retriever = EnsembleRetriever(retrievers=[bm25_retriever, vector_retriever], weights=[0.7, 0.3])


In [9]:
query = "Tesla financial performance and production updates"

Retrived_docs = hybrid_retriever.invoke(query)
print(f"Retrieved {len(Retrived_docs)} documents for the query: '{query}'")
for i, doc in enumerate(Retrived_docs):
    print(f"Document {i+1}: {doc.page_content} (Source: {doc.metadata['source']})")


print(f"Retrieved {len(Retrived_docs)} documents for the query: '{query}'")



Retrieved 29 documents for the query: 'Tesla financial performance and production updates'
Document 1: Tesla announced plans to expand Gigafactory production capacity. (Source: chunk_3)
Document 2: Tesla Cybertruck production ramp begins in 2024 with initial deliveries. (Source: chunk_2)
Document 3: Tesla Model Y became the best-selling vehicle worldwide. (Source: chunk_7)
Document 4: Tesla stock price reached new highs following earnings announcement. (Source: chunk_4)
Document 5: Tesla reported record quarterly revenue of $25.2 billion in Q3 2024. (Source: chunk_0)
Document 6: Tesla reported strong free cash flow generation of $7.5 billion. (Source: chunk_8)
Document 7: The Tesla coil was invented by Nikola Tesla in 1891. (Source: chunk_40)
Document 8: Tesla continues to lead in electric vehicle market share globally. (Source: chunk_6)
Document 9: Electric vehicle charging infrastructure expands rapidly. (Source: chunk_55)
Document 10: Financial markets showed volatility during earni

In [10]:
#Reranking eith cohere
import os
import cohere
documentschupacabra=[]
for doc in Retrived_docs:
    documentschupacabra.append(doc.page_content)
cohere_reranker =cohere.Client() 
reranked_docs = cohere_reranker.rerank(model="rerank-english-v3.0",query=query, documents=documentschupacabra,top_n=10)
print("\nReranked Documents:") 
final = [Retrived_docs[result.index] for result in reranked_docs.results]
for i, doc in enumerate(final):
    print(f"Document {i+1}: {doc.page_content} (Source: {doc.metadata['source']})") 


Reranked Documents:
Document 1: Tesla reported strong free cash flow generation of $7.5 billion. (Source: chunk_8)
Document 2: Tesla reported record quarterly revenue of $25.2 billion in Q3 2024. (Source: chunk_0)
Document 3: Tesla's automotive gross margin improved to 19.3% this quarter. (Source: chunk_1)
Document 4: Tesla announced plans to expand Gigafactory production capacity. (Source: chunk_3)
Document 5: Tesla's energy storage business grew 40% year-over-year. (Source: chunk_5)
Document 6: Tesla continues to lead in electric vehicle market share globally. (Source: chunk_6)
Document 7: Tesla's Full Self-Driving revenue increased significantly. (Source: chunk_9)
Document 8: Tesla Cybertruck production ramp begins in 2024 with initial deliveries. (Source: chunk_2)
Document 9: Tesla stock price reached new highs following earnings announcement. (Source: chunk_4)
Document 10: Financial markets showed volatility during earnings season. (Source: chunk_45)


In [12]:
print(final)

[Document(metadata={'source': 'chunk_8'}, page_content='Tesla reported strong free cash flow generation of $7.5 billion.'), Document(metadata={'source': 'chunk_0'}, page_content='Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.'), Document(id='dec89df2-0d4a-4aa1-a82b-09fe7b8c362a', metadata={'source': 'chunk_1'}, page_content="Tesla's automotive gross margin improved to 19.3% this quarter."), Document(metadata={'source': 'chunk_3'}, page_content='Tesla announced plans to expand Gigafactory production capacity.'), Document(id='09e0dc76-4e69-4f96-9f95-ab620349ee21', metadata={'source': 'chunk_5'}, page_content="Tesla's energy storage business grew 40% year-over-year."), Document(metadata={'source': 'chunk_6'}, page_content='Tesla continues to lead in electric vehicle market share globally.'), Document(id='66e9f075-bcff-429d-b51a-1a994b920cf7', metadata={'source': 'chunk_9'}, page_content="Tesla's Full Self-Driving revenue increased significantly."), Document(metadata=

In [13]:
##generate answer with llm

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
llm = ChatOpenAI(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
llm_input= query + "\n\n" + "\n\n".join([f"{doc.page_content} (Source: {doc.metadata['source']})" for doc in final])
system_message = SystemMessage(content="You are a helpful assistant that provides concise answers based on retrieved documents. When providing an answer, cite each document you base your answer on by their source name.")
human_message = HumanMessage(content=llm_input)

response =llm.invoke([system_message, human_message])
print("\n"+"="*80)
print("Answer from LLM:")
print(response.content)


Answer from LLM:
Here’s a summary of Tesla’s latest financial performance and production updates:

- **Revenue**: Tesla reported record quarterly revenue of $25.2 billion in Q3 2024. (Source: chunk_0)
- **Free Cash Flow**: Tesla generated strong free cash flow of $7.5 billion. (Source: chunk_8)
- **Automotive Gross Margin**: Automotive gross margin improved to 19.3% this quarter. (Source: chunk_1)
- **Energy Storage**: The energy storage business grew 40% year-over-year. (Source: chunk_5)
- **FSD Revenue**: Full Self-Driving revenue increased significantly. (Source: chunk_9)
- **Production Expansion**: Tesla announced plans to expand Gigafactory production capacity. (Source: chunk_3)
- **Cybertruck**: Cybertruck production ramp begins in 2024 with initial deliveries. (Source: chunk_2)
- **Market Position**: Tesla continues to lead in global electric vehicle market share. (Source: chunk_6)
- **Stock Performance**: Tesla’s stock price reached new highs following the earnings announcemen